In [1]:
import os
import pathlib
import datetime
import geoutils as gu

import geomulticorr as gmc
from geomulticorr.correlation import ASP

import matplotlib.pyplot as plt

asp = ASP()

geomulticorr 0.1.0
-------------
telenvi developer version
-------------


/home/cusicand/miniconda3/envs/gmc_env/lib/python3.12/site-packages/richdem/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
session = gmc.open_gmc_session(target_directory_path=pathlib.Path('/home/cusicand/03_Data/GMC_projects/MultiSensor'), epsg_code=2154, empty_geodatabase=True)

[ GeoMultiCorr ✓ ] : GMC project structure found
[ GeoMultiCorr 🚀 ] : Session 'MultiSensor' initialized.


In [3]:
raster_bank = session.map_georasters_bank(f"/home/cusicand/05_Devs/GeoMultiCorr/geomulticorr/data/french_alps/",
                                          epsg_code=2154)

[ GeoMultiCorr 🔍 ] : Mapping georasters
|████████████████████████████████████████| 18/18 [100%] in 4.0s (2.69/s)        


In [8]:
# ! To be tested with a polygon drawn on the map:
get_gdf = session.draw_polygon_manually()

Map(center=[46.8, 8.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_…

Draw your polygon(s) on the map above. When done, run the next cell to get the GeoDataFrame.


In [9]:
# draw on the map, then:
aoi = get_gdf()

In [10]:
session.insert_pzone(aoi.geometry.iloc[0], pz_name="MontVallon", pz_shortname="MV")

[ GeoMultiCorr ✓ ] : Pzone 'MontVallon' inserted.


In [11]:
session.get_pzones_overview()

,pz_name,pz_shortname,pz_surf,geometry
0,MontVallon,MV,None,"POLYGON ((980916.908 6475881.377, 980824.401 6..."


In [12]:
selected_rasters = raster_bank[
    raster_bank["sensor_family"].isin(["spot"]) &
    raster_bank["acq_date"].isin([
        datetime.date(2014, 9, 13),
        datetime.date(2018, 9, 27),
    ])
]
selected_rasters

,filename,sensor,xRes,yRes,bands,rows,cols,acq_date,file_path,sensor_family,sensor_platform,acq_datetime,src_crs,geometry
0,IMG_S6P_2014091336459413CP.tif,spot6,1.5,1.5,1,57599,42999,2014-09-13,/home/cusicand/05_Devs/GeoMultiCorr/geomultico...,spot,spot6,2014-09-13 10:07:38,EPSG:2154,"POLYGON ((969000 6419551.5, 969000 6419568.78,..."
6,IMG_S6P_2018092736618539CP_R1C1.TIF,spot6,1.5,1.5,1,41599,40199,2018-09-27,/home/cusicand/05_Devs/GeoMultiCorr/geomultico...,spot,spot6,2018-09-27 00:00:00,EPSG:2154,"POLYGON ((941700 6454351.5, 941700 6454363.98,..."


In [13]:
session.sieve_bulk(selected_rasters,
                   target_resolution=1.5)                #    canonical_bounds=canon_bounds,
                #    canonical_grid_size=canon_size)

[ GeoMultiCorr 📋 ] : Map epsg : EPSG:2154
Pzone epsg : EPSG:2154
[ GeoMultiCorr 📋 ] : Target resolution: 1.5 (EPSG:2154)
[ GeoMultiCorr 📋 ] : Pzone: MontVallon
[ GeoMultiCorr 📋 ] : Intersecting rasters: 2
Cropping MontVallon_2014-09-13_spot6.tif |██████████████████████████████████████
[ GeoMultiCorr 💾 ] : Saved: /home/cusicand/03_Data/GMC_projects/MultiSensor/raster_data_MultiSensor/MontVallon/opticals/MontVallon_2014-09-13_spot6.tif
Cropping MontVallon_2018-09-27_spot6.tif |██████████████████████████████████████
[ GeoMultiCorr 💾 ] : Saved: /home/cusicand/03_Data/GMC_projects/MultiSensor/raster_data_MultiSensor/MontVallon/opticals/MontVallon_2018-09-27_spot6.tif


{'MontVallon': [PosixPath('/home/cusicand/03_Data/GMC_projects/MultiSensor/raster_data_MultiSensor/MontVallon/opticals/MontVallon_2014-09-13_spot6.tif'),
  PosixPath('/home/cusicand/03_Data/GMC_projects/MultiSensor/raster_data_MultiSensor/MontVallon/opticals/MontVallon_2018-09-27_spot6.tif')]}

In [14]:
session.update_thumbs()

/home/cusicand/miniconda3/envs/gmc_env/lib/python3.12/site-packages/geopandas/array.py:1755: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as RGF93 v1 / Lambert-93 (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))


,th_pz_name,th_path,th_sensor,th_date,th_year,th_valid,th_date_dec,th_date_datetime,geometry
0,MontVallon,/home/cusicand/03_Data/GMC_projects/MultiSenso...,spot6,2014-09-13,2014,0,2014.666667,2014-09-13,"POLYGON ((980823 6475881, 980823 6475881.427, ..."
1,MontVallon,/home/cusicand/03_Data/GMC_projects/MultiSenso...,spot6,2018-09-27,2018,0,2018.666667,2018-09-27,"POLYGON ((980823 6475881, 980823 6475881.427, ..."


## Validate `Thumbs` stage
GeoMultiCorr will only process valid thumbs with `th_valid == 1`. 

In [15]:
session.validate_all_thumbs()

,th_pz_name,th_path,th_sensor,th_date,th_year,th_valid,th_date_dec,th_date_datetime,geometry
0,MontVallon,/home/cusicand/03_Data/GMC_projects/MultiSenso...,spot6,2014-09-13,2014,1,2014.666667,2014-09-13,"POLYGON ((980823 6475881, 980823 6475881.427, ..."
1,MontVallon,/home/cusicand/03_Data/GMC_projects/MultiSenso...,spot6,2018-09-27,2018,1,2018.666667,2018-09-27,"POLYGON ((980823 6475881, 980823 6475881.427, ..."


In [16]:
session.update_pairs_with_strategy(strategy="redundancy", max_step=1)

[ GeoMultiCorr 📋 ] : Updating pairs for PZone: 'MontVallon'


,pa_pz_name,pa_path,pa_left_date,pa_left_sensor,pa_right_date,pa_right_sensor,pa_dt_days,pa_dt_months,pa_dt_years,pa_direction,pa_magn_path,pa_disparity_rd_path,pa_disparity_f_path,pa_status,pa_ew_path,pa_ns_path,pa_cc_path,geometry
0,MontVallon,/home/cusicand/03_Data/GMC_projects/MultiSenso...,2014-09-13,spot6,2018-09-27,spot6,1475,48.46,4.0383,forward,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,empty,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,"POLYGON ((980823 6475881, 980823 6475881.427, ..."
1,MontVallon,/home/cusicand/03_Data/GMC_projects/MultiSenso...,2018-09-27,spot6,2014-09-13,spot6,1475,48.46,4.0383,backward,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,empty,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,"POLYGON ((980823 6475881, 980823 6475881.427, ..."


In [18]:
scripts = session.prepare_pairs_correlation(cluster="local",
                                            cores=16,
                                            corr_algorithm="asp_bm",
                                            corr_kernel=(21, 21),
                                            overwrite_scripts=True)

[ GeoMultiCorr 📋 ] : Preparing 2 pair(s) for correlation [cluster=local]
[ GeoMultiCorr 📋 ] : Ready: 'MontVallon_2014-09-13-spot6_2018-09-27-spot6' → MontVallon_2014-09-13-spot6_2018-09-27-spot6_CorrelationJob.sh
[ GeoMultiCorr 📋 ] : Ready: 'MontVallon_2018-09-27-spot6_2014-09-13-spot6' → MontVallon_2018-09-27-spot6_2014-09-13-spot6_CorrelationJob.sh
[ GeoMultiCorr ✓ ] : Prepared 2 bash script(s) in their respective pair directories.


In [19]:
session.launch_pairs_correlation()

[ GeoMultiCorr 📋 ] : Preparing 2 pair(s) for correlation [cluster=local]
[ GeoMultiCorr 📋 ] : Reusing existing script: MontVallon_2014-09-13-spot6_2018-09-27-spot6_CorrelationJob.sh
[ GeoMultiCorr 📋 ] : Reusing existing script: MontVallon_2018-09-27-spot6_2014-09-13-spot6_CorrelationJob.sh
[ GeoMultiCorr ✓ ] : Prepared 2 bash script(s) in their respective pair directories.
[ GeoMultiCorr 📋 ] : Launching: MontVallon_2014-09-13-spot6_2018-09-27-spot6_CorrelationJob.sh
Deleted 28 file(s) from '/home/cusicand/03_Data/GMC_projects/MultiSensor/raster_data_MultiSensor/MontVallon/image_correlation/MontVallon_2014-09-13-spot6_2018-09-27-spot6'.
[ GeoMultiCorr 📋 ] : Launching: MontVallon_2018-09-27-spot6_2014-09-13-spot6_CorrelationJob.sh
Deleted 28 file(s) from '/home/cusicand/03_Data/GMC_projects/MultiSensor/raster_data_MultiSensor/MontVallon/image_correlation/MontVallon_2018-09-27-spot6_2014-09-13-spot6'.
[ GeoMultiCorr ✓ ] : Launched 2/2 pair(s) successfully.
[ GeoMultiCorr 💾 ] : Geodatabase

[0, 0]

In [20]:
session.get_pairs_overview()

,pa_pz_name,pa_path,pa_left_date,pa_left_sensor,pa_right_date,pa_right_sensor,pa_dt_days,pa_dt_months,pa_dt_years,pa_direction,pa_magn_path,pa_disparity_rd_path,pa_disparity_f_path,pa_status,pa_ew_path,pa_ns_path,pa_cc_path,geometry
0,MontVallon,/home/cusicand/03_Data/GMC_projects/MultiSenso...,2014-09-13,spot6,2018-09-27,spot6,1475,48.46,4.0383,forward,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,complete,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,"POLYGON ((980823 6475881, 980823 6475881.427, ..."
1,MontVallon,/home/cusicand/03_Data/GMC_projects/MultiSenso...,2018-09-27,spot6,2014-09-13,spot6,1475,48.46,4.0383,backward,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,complete,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,/home/cusicand/03_Data/GMC_projects/MultiSenso...,"POLYGON ((980823 6475881, 980823 6475881.427, ..."


In [24]:
session.extract_pairs_raw_displacements(save_plot=True, overwrite=True)

[ GeoMultiCorr 🚀 ] : Extracting raw displacements for 'MontVallon_2014-09-13-spot6_2018-09-27-spot6'


/home/cusicand/miniconda3/envs/gmc_env/lib/python3.12/site-packages/geoutils/raster/raster.py:1860: UserWarning: Unmasked values equal to the nodata value found in data array. They are now masked.
 If this happened when creating or updating the array, to silence this warning, convert nodata values in the array to np.nan or mask them with np.ma.masked prior to creating or updating the raster.
If this happened during a numerical operation, use astype() prior to the operation to convert to a data type that won't derive the nodata values (e.g., a float type).
  warnings.warn(


[ GeoMultiCorr 📄 ] : Saved EW displacement: MontVallon_2014-09-13-spot6_2018-09-27-spot6-F_EW.tif
[ GeoMultiCorr 📄 ] : Saved NS displacement: MontVallon_2014-09-13-spot6_2018-09-27-spot6-F_NS.tif
[ GeoMultiCorr 📊 ] : Raw correlation stats saved for 'MontVallon_2014-09-13-spot6_2018-09-27-spot6' (5 raster(s) processed)
[ GeoMultiCorr 💾 ] : Control plot saved: MontVallon_2014-09-13-spot6_2018-09-27-spot6_raw_disp.jpg
[ GeoMultiCorr 🚀 ] : Extracting raw displacements for 'MontVallon_2018-09-27-spot6_2014-09-13-spot6'


/home/cusicand/miniconda3/envs/gmc_env/lib/python3.12/site-packages/geoutils/raster/raster.py:1860: UserWarning: Unmasked values equal to the nodata value found in data array. They are now masked.
 If this happened when creating or updating the array, to silence this warning, convert nodata values in the array to np.nan or mask them with np.ma.masked prior to creating or updating the raster.
If this happened during a numerical operation, use astype() prior to the operation to convert to a data type that won't derive the nodata values (e.g., a float type).
  warnings.warn(


[ GeoMultiCorr 📄 ] : Saved EW displacement: MontVallon_2018-09-27-spot6_2014-09-13-spot6-F_EW.tif
[ GeoMultiCorr 📄 ] : Saved NS displacement: MontVallon_2018-09-27-spot6_2014-09-13-spot6-F_NS.tif
[ GeoMultiCorr 📊 ] : Raw correlation stats saved for 'MontVallon_2018-09-27-spot6_2014-09-13-spot6' (5 raster(s) processed)
[ GeoMultiCorr 💾 ] : Control plot saved: MontVallon_2018-09-27-spot6_2014-09-13-spot6_raw_disp.jpg
[ GeoMultiCorr ✓ ] : Extracted raw displacements for 2/2 pair(s) (0 skipped).


{'MontVallon_2014-09-13-spot6_2018-09-27-spot6': {'metadata': {'pair_key': 'MontVallon_2014-09-13-spot6_2018-09-27-spot6',
   'pair_name': 'MontVallon_2014-09-13-spot6_2018-09-27-spot6',
   'pzone': 'MontVallon',
   'left_date': '2014-09-13',
   'left_sensor': 'spot6',
   'right_date': '2018-09-27',
   'right_sensor': 'spot6',
   'dt_days': 1475,
   'dt_months': 48.46,
   'dt_years': 4.0383,
   'direction': 'forward',
   'ncols': 1514,
   'nrows': 1424,
   'resolution_x': 1.5,
   'resolution_y': 1.5,
   'crs': 'EPSG:2154',
   'bounds': {'left': 980823.0,
    'bottom': 6475881.0,
    'right': 983094.0,
    'top': 6478017.0}},
  'raw_corr_stats': {'ew': {'count_total': 2155936,
    'count_valid': 2019212,
    'valid_fraction': 0.936583,
    'mean': -0.554911,
    'median': -0.590154,
    'std': 0.772055,
    'nmad': 0.365538,
    'min': -14.523896,
    'max': 20.78629,
    'p5': -1.274684,
    'p25': -0.830476,
    'p50': -0.590154,
    'p75': -0.336387,
    'p95': 0.286682},
   'ns': {'